# Week 6 — Day 1: Data Curation

## "The Price is Right" Capstone Project

Build a model that predicts how much a product costs from its description, using a curated scrape of Amazon data.

**Plan for the week:**
- **Day 1** — Data curation (this notebook)
- **Day 2** — Data pre-processing (rewrite product descriptions to a standard format using LLMs)
- **Day 3** — Evaluation, baselines, traditional ML
- **Day 4** — Deep learning + frontier LLMs (zero-shot)
- **Day 5** — Fine-tuning a frontier model

Source dataset: https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023 — specifically the `raw/meta_categories` folder for product metadata.

Today: load the raw data, filter to items with usable prices and descriptions, balance across categories, and push the curated dataset to the HuggingFace Hub.

## Setup

In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from dotenv import load_dotenv
from huggingface_hub import login
from datasets import load_dataset
from tqdm.notebook import tqdm
from pricer.items import Item
from pricer.parser import parse

load_dotenv(override=True)

In [ ]:
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

## Load a single category as a starter

If `load_dataset` errors with `trust_remote_code is no longer supported`, run `!uv add --upgrade datasets==3.6.0` in a new cell, restart the kernel, and try again.

In [ ]:
dataset = load_dataset(
    "McAuley-Lab/Amazon-Reviews-2023",
    "raw_meta_Appliances",
    split="full",
    trust_remote_code=True,
)

In [ ]:
print(f"Number of Appliances: {len(dataset):,}")

In [ ]:
dataset[6]

In [ ]:
# What's the most expensive item?
max_price = 0
max_item = None

for datapoint in tqdm(dataset):
    try:
        price = float(datapoint["price"])
        if price > max_price:
            max_item = datapoint
            max_price = price
    except ValueError:
        pass

print(f"The most expensive item is {max_item['title']} and it costs {max_price:,.2f}")

## Parse into Item objects

`parse(...)` keeps items priced $1-$1000 with sufficient description detail.

In [ ]:
items = [parse(datapoint, "Appliances") for datapoint in tqdm(dataset)]
items = [item for item in items if item is not None]
print(f"There are {len(items):,} items from {len(dataset):,} datapoints")

In [ ]:
items[0]

In [ ]:
print(items[0].full)

## Explore distributions

In [ ]:
prices = [item.price for item in items]
lengths = [len(item.full) for item in items]

In [ ]:
plt.figure(figsize=(15, 6))
plt.title(f"Lengths: Avg {sum(lengths)/len(lengths):,.0f} and highest {max(lengths):,}\n")
plt.xlabel('Length (chars)')
plt.ylabel('Count')
plt.hist(lengths, rwidth=0.7, color="lightblue", bins=range(0, 6000, 100))
plt.show()

In [ ]:
max_length = max(lengths)
max_length_item = items[lengths.index(max_length)]
print(max_length_item.full)

In [ ]:
plt.figure(figsize=(15, 6))
plt.title(f"Prices: Avg {sum(prices)/len(prices):,.2f} and highest {max(prices):,}\n")
plt.xlabel('Price ($)')
plt.ylabel('Count')
plt.hist(prices, rwidth=0.7, color="orange", bins=range(0, 1000, 10))
plt.show()

## Load all 8 categories using ItemLoader

Wraps the load + parse + filter logic for any category.

In [ ]:
from pricer.loaders import ItemLoader
loader = ItemLoader("Appliances")
items = loader.load()

In [ ]:
dataset_names = [
    "Automotive",
    "Electronics",
    "Office_Products",
    "Tools_and_Home_Improvement",
    "Cell_Phones_and_Accessories",
    "Toys_and_Games",
    "Appliances",
    "Musical_Instruments",
]

In [ ]:
items = []
for dataset_name in dataset_names:
    loader = ItemLoader(dataset_name)
    items.extend(loader.load())

In [ ]:
print(f"A grand total of {len(items):,} items")

## Deduplicate by title and full text

In [ ]:
random.seed(42)
random.shuffle(items)

seen = set()
items = [x for x in tqdm(items) if not (x.title in seen or seen.add(x.title))]

seen = set()
items = [x for x in tqdm(items) if not (x.full in seen or seen.add(x.full))]

del seen
print(f"After deduplication, we have {len(items):,} items")

In [ ]:
lengths = [len(item.full) for item in items]
plt.figure(figsize=(15, 6))
plt.title(f"Text length: Avg {sum(lengths)/len(lengths):,.1f} and highest {max(lengths):,}\n")
plt.xlabel('Length (characters)')
plt.ylabel('Count')
plt.hist(lengths, rwidth=0.7, color="skyblue", bins=range(0, 4050, 50))
plt.show()

In [ ]:
prices = [item.price for item in items]
plt.figure(figsize=(15, 6))
plt.title(f"Prices: Avg {sum(prices)/len(prices):,.1f} and highest {max(prices):,}\n")
plt.xlabel('Price ($)')
plt.ylabel('Count')
plt.hist(prices, rwidth=0.7, color="blueviolet", bins=range(0, 1000, 10))
plt.show()

In [ ]:
category_counts = Counter([item.category for item in items])
categories = category_counts.keys()
counts = [category_counts[c] for c in categories]

plt.figure(figsize=(15, 6))
plt.bar(categories, counts, color="goldenrod")
plt.title('How many in each category')
plt.xlabel('Categories')
plt.ylabel('Count')
plt.xticks(rotation=30, ha='right')
for i, v in enumerate(counts):
    plt.text(i, v, f"{v:,}", ha='center', va='bottom')
plt.show()

## Weighted sampling for a more balanced dataset

The raw distribution is dominated by cheap items and Automotive. Weight by price² and downweight the over-represented categories so the model sees a wider, more interesting price distribution.

In [ ]:
np.random.seed(42)

SIZE = 820_000

prices_arr = np.array([it.price for it in items], dtype=float)
categories_arr = np.array([it.category for it in items])
p = (prices_arr - prices_arr.min()) / (prices_arr.max() - prices_arr.min() + 1e-9)

w = p ** 2
w[categories_arr == "Tools_and_Home_Improvement"] *= 0.5
w[categories_arr == "Automotive"] *= 0.05

w = w / w.sum()
idx = np.random.choice(len(items), size=SIZE, replace=False, p=w)
sample = [items[i] for i in idx]

In [ ]:
prices = [item.price for item in sample]
plt.figure(figsize=(15, 6))
plt.title(f"Prices: Avg {sum(prices)/len(prices):,.1f} lowest {min(prices):,} and highest {max(prices):,}\n")
plt.xlabel('Price ($)')
plt.ylabel('Count')
plt.hist(prices, rwidth=0.7, color="blueviolet", bins=range(0, 1000, 10))
plt.show()

In [ ]:
random.seed(42)
random.shuffle(sample)

In [ ]:
category_counts = Counter([item.category for item in sample])
categories = category_counts.keys()
counts = [category_counts[c] for c in categories]

plt.figure(figsize=(15, 6))
plt.bar(categories, counts, color="goldenrod")
plt.title('How many in each category')
plt.xlabel('Categories')
plt.ylabel('Count')
plt.xticks(rotation=30, ha='right')
for i, v in enumerate(counts):
    plt.text(i, v, f"{v:,}", ha='center', va='bottom')
plt.show()

In [ ]:
plt.figure(figsize=(12, 10))
plt.pie(counts, labels=categories, autopct='%1.0f%%', startangle=90)
centre_circle = plt.Circle((0, 0), 0.70, fc='white')
fig = plt.gcf()
fig.gca().add_artist(centre_circle)
plt.title('Categories')
plt.axis('equal')
plt.show()

In [ ]:
sizes = [len(item.full) for item in sample]
prices = [item.price for item in sample]

plt.figure(figsize=(15, 8))
plt.scatter(sizes, prices, s=0.2, color="red")
plt.xlabel('Size')
plt.ylabel('Price')
plt.title('Is there a simple correlation with text length?')
plt.show()

In [ ]:
ounces = [item.weight for item in sample]
prices = [item.price for item in sample]

plt.figure(figsize=(15, 8))
plt.scatter(ounces, prices, s=0.2, color="darkorange")
plt.xlabel('Weight (ounces)')
plt.ylabel('Price')
plt.xlim(0, 400)
plt.title('Is there a simple correlation with weight?')
plt.show()

## Push the curated dataset to the HuggingFace Hub

Replace `YOUR_HF_USERNAME` with your own Hugging Face username before running.

In [ ]:
username = "YOUR_HF_USERNAME"  # replace with your Hugging Face username
full = f"{username}/items_raw_full"
lite = f"{username}/items_raw_lite"

train = sample[:800_000]
val = sample[800_000:810_000]
test = sample[810_000:]

Item.push_to_hub(full, train, val, test)

train_lite = train[:20_000]
val_lite = val[:1_000]
test_lite = test[:1_000]
Item.push_to_hub(lite, train_lite, val_lite, test_lite)